# Trader Bias Analysis
Exploratory analysis comparing behavioral patterns across 4 trader types:
**Calm**, **Loss Averse**, **Overtrader**, **Revenge Trader**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

COLORS = {
    'calm':        '#2ecc71',
    'loss_averse': '#e74c3c',
    'overtrader':  '#f39c12',
    'revenge':     '#9b59b6',
}

LABELS = {
    'calm':        'Calm',
    'loss_averse': 'Loss Averse',
    'overtrader':  'Overtrader',
    'revenge':     'Revenge',
}

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333333',
    'axes.labelcolor':  '#cccccc',
    'xtick.color':      '#cccccc',
    'ytick.color':      '#cccccc',
    'text.color':       '#cccccc',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
})

print('Libraries loaded.')

Matplotlib is building the font cache; this may take a moment.


Libraries loaded.


In [2]:
FILE_MAP = {
    'calm':        'trading_datasets/calm_trader.csv',
    'loss_averse': 'trading_datasets/loss_averse_trader.csv',
    'overtrader':  'trading_datasets/overtrader.csv',
    'revenge':     'trading_datasets/revenge_trader.csv',
}

raw = {}
for name, fpath in FILE_MAP.items():
    df = pd.read_csv(fpath, parse_dates=['timestamp'])
    df = df.dropna(subset=['profit_loss', 'quantity', 'entry_price', 'exit_price', 'balance'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['trader_type'] = name
    raw[name] = df

for name, df in raw.items():
    print(f'{LABELS[name]:12s}: {len(df):,} trades  |  {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')

FileNotFoundError: [Errno 2] No such file or directory: 'trading_datasets/calm_trader.csv'

## 1. Summary Statistics

In [ ]:
rows = []
for name, df in raw.items():
    wins   = df[df['profit_loss'] > 0]
    losses = df[df['profit_loss'] < 0]
    df['time_diff'] = df['timestamp'].diff().dt.total_seconds()

    df['prev_pl'] = df['profit_loss'].shift(1)
    size_after_loss = df[df['prev_pl'] < 0]['quantity'].mean()
    size_after_win  = df[df['prev_pl'] > 0]['quantity'].mean()

    df['price_move_pct'] = (df['exit_price'] - df['entry_price']).abs() / df['entry_price'] * 100
    hold_wins   = df[df['profit_loss'] > 0]['price_move_pct'].mean()
    hold_losses = df[df['profit_loss'] < 0]['price_move_pct'].mean()

    rows.append({
        'Trader':               LABELS[name],
        'Win Rate':             f"{len(wins)/len(df):.1%}",
        'Avg Win ($)':          f"{wins['profit_loss'].mean():.1f}",
        'Avg Loss ($)':         f"{losses['profit_loss'].mean():.1f}",
        'Loss/Win Ratio':       f"{abs(losses['profit_loss'].mean()) / wins['profit_loss'].mean():.2f}x",
        'Avg Seconds/Trade':    f"{df['time_diff'].mean():.1f}s",
        'Size After Loss/Win':  f"{size_after_loss/size_after_win:.2f}x",
        'Hold Asymmetry':       f"{hold_losses/hold_wins:.2f}x",
        'Final Balance ($)':    f"{df['balance'].iloc[-1]:,.0f}",
    })

summary = pd.DataFrame(rows).set_index('Trader')
summary

## 2. Account Balance Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for name, df in raw.items():
    ax.plot(df['timestamp'], df['balance'],
            color=COLORS[name], label=LABELS[name], linewidth=1.2, alpha=0.9)

ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
ax.set_title('Account Balance Over Time', fontsize=14, pad=12)
ax.set_xlabel('Date')
ax.set_ylabel('Balance ($)')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(True)
plt.tight_layout()
plt.show()

## 3. Win Rate & P/L Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Win rate bar
ax = axes[0]
names = list(raw.keys())
win_rates = [len(raw[n][raw[n]['profit_loss'] > 0]) / len(raw[n]) * 100 for n in names]
bars = ax.bar([LABELS[n] for n in names], win_rates,
              color=[COLORS[n] for n in names], width=0.5, edgecolor='#333')
ax.axhline(50, color='white', linewidth=0.8, linestyle='--', alpha=0.5, label='50% baseline')
for bar, val in zip(bars, win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_title('Win Rate by Trader Type', fontsize=13)
ax.set_ylabel('Win Rate (%)')
ax.set_ylim(0, 75)
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True, axis='y')

# P/L distribution (clipped for readability)
ax = axes[1]
for name, df in raw.items():
    clipped = df['profit_loss'].clip(-500, 500)
    clipped.plot.kde(ax=ax, color=COLORS[name], label=LABELS[name], linewidth=2)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('P/L Distribution (clipped ±$500)', fontsize=13)
ax.set_xlabel('Profit / Loss ($)')
ax.set_ylabel('Density')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True)

plt.tight_layout()
plt.show()

## 4. Avg Win vs Avg Loss — The Loss Aversion Signal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(raw.keys())
avg_wins   = [raw[n][raw[n]['profit_loss'] > 0]['profit_loss'].mean() for n in names]
avg_losses = [abs(raw[n][raw[n]['profit_loss'] < 0]['profit_loss'].mean()) for n in names]

# Side by side bars
ax = axes[0]
x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, avg_wins,   width=w, label='Avg Win',  color='#2ecc71', edgecolor='#333')
ax.bar(x + w/2, avg_losses, width=w, label='Avg Loss', color='#e74c3c', edgecolor='#333')
ax.set_xticks(x)
ax.set_xticklabels([LABELS[n] for n in names])
ax.set_title('Avg Win vs Avg Loss Size', fontsize=13)
ax.set_ylabel('$ Amount')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.set_yscale('log')
ax.grid(True, axis='y')
ax.set_ylim(bottom=1)

# Loss/Win ratio
ax = axes[1]
ratios = [l/w for l, w in zip(avg_losses, avg_wins)]
bars = ax.bar([LABELS[n] for n in names], ratios,
              color=[COLORS[n] for n in names], width=0.5, edgecolor='#333')
ax.axhline(1, color='white', linewidth=0.8, linestyle='--', alpha=0.5, label='1:1 baseline')
for bar, val in zip(bars, ratios):
    label = f'{val:.1f}x' if val < 10 else f'{val:.0f}x'
    ax.text(bar.get_x() + bar.get_width()/2, min(bar.get_height() * 1.02, ax.get_ylim()[1] * 0.9),
            label, ha='center', va='bottom', fontsize=10)
ax.set_title('Loss / Win Ratio (higher = worse)', fontsize=13)
ax.set_ylabel('Ratio')
ax.set_yscale('log')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 5. Trade Frequency — The Overtrading Signal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg seconds between trades
ax = axes[0]
time_diffs = {}
for name, df in raw.items():
    diff = df['timestamp'].diff().dt.total_seconds().dropna()
    time_diffs[name] = diff

avgs = [time_diffs[n].mean() for n in names]
bars = ax.bar([LABELS[n] for n in names], avgs,
              color=[COLORS[n] for n in names], width=0.5, edgecolor='#333')
for bar, val in zip(bars, avgs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}s', ha='center', va='bottom', fontsize=10)
ax.set_title('Avg Seconds Between Trades', fontsize=13)
ax.set_ylabel('Seconds')
ax.grid(True, axis='y')

# Trades per hour heatmap
ax = axes[1]
for name, df in raw.items():
    hourly = df.set_index('timestamp').resample('1h').size()
    hourly.plot(ax=ax, color=COLORS[name], label=LABELS[name], linewidth=1, alpha=0.8)
ax.set_title('Trades per Hour Over Time', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('# Trades')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. Revenge Trading — Size Escalation After Losses

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compute loss streaks for each trader
def compute_streaks(df):
    streaks = []
    streak = 0
    for pl in df['profit_loss']:
        if pl < 0:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    return streaks

ax = axes[0]
streak_levels = [0, 1, 2, 3, 4, 5]
for name, df in raw.items():
    df = df.copy()
    df['loss_streak'] = compute_streaks(df)
    means = [df[df['loss_streak'] == s]['quantity'].mean() for s in streak_levels]
    # Normalize to streak=0 baseline
    baseline = means[0] if means[0] > 0 else 1
    normalized = [m / baseline for m in means]
    ax.plot(streak_levels, normalized, color=COLORS[name], label=LABELS[name],
            linewidth=2, marker='o', markersize=5)

ax.axhline(1.0, color='white', linewidth=0.8, linestyle='--', alpha=0.4, label='baseline')
ax.set_title('Normalized Trade Size vs Loss Streak Depth', fontsize=13)
ax.set_xlabel('Consecutive Losses (streak depth)')
ax.set_ylabel('Relative Trade Size (1.0 = baseline)')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True)

# Size after loss vs after win
ax = axes[1]
size_after_loss = []
size_after_win  = []
for name, df in raw.items():
    df = df.copy()
    df['prev_pl'] = df['profit_loss'].shift(1)
    size_after_loss.append(df[df['prev_pl'] < 0]['quantity'].mean())
    size_after_win.append(df[df['prev_pl'] > 0]['quantity'].mean())

x = np.arange(len(names))
w = 0.35
ax.bar(x - w/2, size_after_win,  width=w, label='After Win',  color='#2ecc71', edgecolor='#333')
ax.bar(x + w/2, size_after_loss, width=w, label='After Loss', color='#e74c3c', edgecolor='#333')
ax.set_xticks(x)
ax.set_xticklabels([LABELS[n] for n in names])
ax.set_title('Avg Trade Size: After Win vs After Loss', fontsize=13)
ax.set_ylabel('Avg Quantity')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 7. Hold Time Asymmetry — Cutting Winners Short, Riding Losers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, df in raw.items():
    df = df.copy()
    df['price_move_pct'] = (df['exit_price'] - df['entry_price']).abs() / df['entry_price'] * 100
    wins_move   = df[df['profit_loss'] > 0]['price_move_pct'].clip(0, 2)
    losses_move = df[df['profit_loss'] < 0]['price_move_pct'].clip(0, 2)
    wins_move.plot.kde(ax=ax, color=COLORS[name], linewidth=2,
                       label=f'{LABELS[name]} wins', linestyle='-')
    losses_move.plot.kde(ax=ax, color=COLORS[name], linewidth=1.5,
                         label=f'{LABELS[name]} losses', linestyle='--', alpha=0.6)

ax.set_title('Price Move % on Wins vs Losses (clipped 0-2%)', fontsize=13)
ax.set_xlabel('Abs Price Move (%)')
ax.set_ylabel('Density')
ax.legend(facecolor='#1a1a1a', edgecolor='#444', fontsize=7)
ax.grid(True)

# Hold asymmetry ratio bar
ax = axes[1]
asymmetry = []
for name, df in raw.items():
    df = df.copy()
    df['price_move_pct'] = (df['exit_price'] - df['entry_price']).abs() / df['entry_price'] * 100
    hold_wins   = df[df['profit_loss'] > 0]['price_move_pct'].mean()
    hold_losses = df[df['profit_loss'] < 0]['price_move_pct'].mean()
    asymmetry.append(hold_losses / hold_wins)

bars = ax.bar([LABELS[n] for n in names], asymmetry,
              color=[COLORS[n] for n in names], width=0.5, edgecolor='#333')
ax.axhline(1.0, color='white', linewidth=0.8, linestyle='--', alpha=0.5, label='symmetric (1.0)')
for bar, val in zip(bars, asymmetry):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}x', ha='center', va='bottom', fontsize=10)
ax.set_title('Hold Asymmetry: Loss Move / Win Move\n(>1 = holds losses longer)', fontsize=13)
ax.set_ylabel('Ratio')
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 8. Bias Signal Heatmap — Feature Correlation per Trader

In [ ]:
def extract_features(df):
    df = df.copy()
    df['prev_pl']    = df['profit_loss'].shift(1)
    df['time_diff']  = df['timestamp'].diff().dt.total_seconds()
    df['price_move'] = (df['exit_price'] - df['entry_price']).abs() / df['entry_price'] * 100

    wins   = df[df['profit_loss'] > 0]
    losses = df[df['profit_loss'] < 0]

    avg_win        = wins['profit_loss'].mean()
    avg_loss       = abs(losses['profit_loss'].mean())
    hold_wins      = wins['price_move'].mean()
    hold_losses    = losses['price_move'].mean()
    size_after_loss= df[df['prev_pl'] < 0]['quantity'].mean()
    size_after_win = df[df['prev_pl'] > 0]['quantity'].mean()

    return {
        'Win Rate':            len(wins) / len(df),
        'Loss/Win Ratio':      avg_loss / avg_win,
        'Hold Asymmetry':      hold_losses / hold_wins,
        'Avg Secs/Trade':      df['time_diff'].mean(),
        'Size Escal. (Loss)':  size_after_loss / size_after_win,
        'Quantity Volatility': df['quantity'].std() / df['quantity'].mean(),
        'P/L Volatility':      df['profit_loss'].std() / abs(df['profit_loss'].mean() + 1e-9),
    }

feature_df = pd.DataFrame(
    {LABELS[n]: extract_features(df) for n, df in raw.items()}
).T

# Normalize each feature 0-1 for heatmap
norm = (feature_df - feature_df.min()) / (feature_df.max() - feature_df.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(norm, annot=feature_df.round(2), fmt='g',
            cmap='RdYlGn_r', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Normalized Score (higher = more extreme)'},
            annot_kws={'size': 9})
ax.set_title('Bias Signal Heatmap — Raw Values, Color-Normalized', fontsize=13, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 9. Radar Chart — Bias Profile per Trader

In [ ]:
from matplotlib.patches import FancyArrowPatch

bias_scores_raw = {
    LABELS[n]: {
        'Overtrading':    1 / extract_features(df)['Avg Secs/Trade'],   # more trades = higher
        'Loss Aversion':  extract_features(df)['Loss/Win Ratio'],
        'Hold Asymmetry': extract_features(df)['Hold Asymmetry'],
        'Revenge':        extract_features(df)['Size Escal. (Loss)'],
        'Volatility':     extract_features(df)['Quantity Volatility'],
    }
    for n, df in raw.items()
}

bias_df = pd.DataFrame(bias_scores_raw).T
norm_bias = (bias_df - bias_df.min()) / (bias_df.max() - bias_df.min() + 1e-9)

categories = list(norm_bias.columns)
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw=dict(polar=True))
fig.suptitle('Bias Radar — Per Trader Type', fontsize=14, y=1.02)

for ax, (trader, row) in zip(axes, norm_bias.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    color = COLORS[[k for k, v in LABELS.items() if v == trader][0]]

    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['', '', '', ''], size=6)
    ax.set_facecolor('#1a1a1a')
    ax.spines['polar'].set_color('#333')
    ax.grid(color='#2a2a2a')
    ax.set_title(trader, size=11, pad=12, color=color)

plt.tight_layout()
plt.show()

## 10. Cumulative P/L & Drawdown

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

ax = axes[0]
for name, df in raw.items():
    cumpl = df['profit_loss'].cumsum()
    ax.plot(range(len(cumpl)), cumpl,
            color=COLORS[name], label=LABELS[name], linewidth=1.2)
ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
ax.set_title('Cumulative P/L (trade-by-trade)', fontsize=13)
ax.set_ylabel('Cumulative P/L ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True)

ax = axes[1]
for name, df in raw.items():
    cumpl    = df['profit_loss'].cumsum()
    rolling_max = cumpl.cummax()
    drawdown = cumpl - rolling_max
    ax.fill_between(range(len(drawdown)), drawdown, 0,
                    color=COLORS[name], alpha=0.4, label=LABELS[name])
    ax.plot(range(len(drawdown)), drawdown, color=COLORS[name], linewidth=0.8)
ax.set_title('Drawdown (from peak)', fontsize=13)
ax.set_xlabel('Trade #')
ax.set_ylabel('Drawdown ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(facecolor='#1a1a1a', edgecolor='#444')
ax.grid(True)

plt.tight_layout()
plt.show()

## 11. Key Takeaways

In [ ]:
print('=== KEY SIGNALS PER BIAS ===')
print()
print('CALM TRADER')
print('  - Win rate ~50%, avg win ≈ avg loss (1:1 ratio)')
print('  - Consistent trade size regardless of prior outcome')
print('  - Symmetric hold time on wins and losses')
print()
print('LOSS AVERSE')
print('  - Win rate >60% but avg loss is 231x avg win')
print('  - Cuts winners early, lets losers run 3.35x longer')
print('  - Key feature: loss/win ratio and hold asymmetry')
print()
print('OVERTRADER')
print('  - Trades every 10s vs 61s for others (6x faster)')
print('  - Win rate and P/L per trade are normal — just too many')
print('  - Key feature: avg seconds between trades')
print()
print('REVENGE TRADER')
print('  - Trade size escalates with loss streak depth')
print('  - Higher quantity volatility (std/mean)')
print('  - Key feature: size after loss vs after win, streak-size correlation')